# 🚇 BheedMitra Production Dataset Generator

## Production-Quality Metro Transit Time Series Data

This notebook generates realistic, station-wise hourly datasets for:
- 🔮 **Crowd Prediction**
- 🚦 **Congestion Analysis**
- 🔍 **Anomaly Detection**

### Features:
- ✅ GTFS integration (real transit data)
- ✅ Weather API (Open-Meteo)
- ✅ Realistic ridership simulation
- ✅ Feature engineering for ML
- ✅ Data validation
- ✅ EDA visualizations

---

## 1️⃣ Setup & Installation

In [ ]:
# Install required packages
!pip install pandas numpy requests matplotlib seaborn -q

print("✅ Packages installed successfully!")

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import os
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Libraries imported successfully!")

## 2️⃣ Upload Pipeline Script

**Option A**: Upload `bheedmitra_production_pipeline.py` from your local machine

**Option B**: Copy-paste the code in the next cell

In [ ]:
# Download the pipeline script from GitHub or paste the code here
# For this example, we'll assume you've uploaded bheedmitra_production_pipeline.py

# If you uploaded the file:
from bheedmitra_production_pipeline import (
    generate_dataset,
    DMRC_CONFIG,
    MTA_CONFIG,
    CityConfig
)

print("✅ Pipeline loaded!")

## 3️⃣ Generate DMRC (Delhi Metro) Dataset

This will generate a comprehensive dataset for Delhi Metro with:
- 200+ stations across 8 lines
- 30 days of hourly data (720 hours × 200 stations = 144,000 records)
- Realistic ridership patterns
- Weather integration

In [ ]:
# Generate DMRC dataset
dmrc_df = generate_dataset(
    config=DMRC_CONFIG,
    start_date=datetime(2024, 1, 1),
    end_date=datetime(2024, 1, 31),
    output_dir="output"
)

## 4️⃣ Explore the Generated Dataset

In [ ]:
# Display first few rows
print("📊 Sample Data:")
print("="*80)
dmrc_df.head(10)

In [ ]:
# Dataset info
print("📈 Dataset Information:")
print("="*80)
dmrc_df.info()

In [ ]:
# Statistical summary
print("📊 Statistical Summary:")
print("="*80)
dmrc_df.describe()

In [ ]:
# Column-wise breakdown
print("\n🔍 Dataset Breakdown:")
print("="*80)
print(f"Total Records:      {len(dmrc_df):,}")
print(f"Total Stations:     {dmrc_df['station_id'].nunique()}")
print(f"Total Routes:       {dmrc_df['route_id'].nunique()}")
print(f"Date Range:         {dmrc_df['timestamp'].min()} to {dmrc_df['timestamp'].max()}")
print(f"\nRidership Stats:")
print(f"  Mean:             {dmrc_df['ridership_count'].mean():.0f} people/hour")
print(f"  Median:           {dmrc_df['ridership_count'].median():.0f} people/hour")
print(f"  Peak:             {dmrc_df['ridership_count'].max():,} people/hour")
print(f"\nCongestion Distribution:")
print(dmrc_df['crowd_level'].value_counts())
print(f"\nPeak vs Off-Peak:")
print(dmrc_df.groupby('is_peak_hour')['ridership_count'].mean())

## 5️⃣ Visualizations

Explore the generated plots in the output folder:

In [ ]:
# Display generated plots
from IPython.display import Image, display

print("📊 Ridership Analysis:")
display(Image('output/ridership_analysis.png'))

print("\n🔥 Station Heatmap:")
display(Image('output/ridership_heatmap.png'))

print("\n🌤️ Weather Impact:")
display(Image('output/weather_impact.png'))

## 6️⃣ Custom Analysis

Perform your own analysis:

In [ ]:
# Top 10 busiest stations
top_stations = dmrc_df.groupby('station_name')['ridership_count'].sum().nlargest(10)

plt.figure(figsize=(12, 6))
top_stations.plot(kind='barh', color='#2E86AB')
plt.xlabel('Total Ridership', fontsize=12)
plt.ylabel('Station', fontsize=12)
plt.title('Top 10 Busiest Stations (30-Day Total)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Weekday vs Weekend comparison
weekday_weekend = dmrc_df.groupby(['is_weekend', 'hour_of_day'])['ridership_count'].mean().unstack(0)
weekday_weekend.columns = ['Weekday', 'Weekend']

plt.figure(figsize=(14, 6))
weekday_weekend.plot(marker='o', linewidth=2)
plt.xlabel('Hour of Day', fontsize=12)
plt.ylabel('Average Ridership', fontsize=12)
plt.title('Weekday vs Weekend Ridership Patterns', fontsize=14, fontweight='bold')
plt.legend(title='Day Type', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Route-wise congestion comparison
route_congestion = dmrc_df.groupby('route_id').agg({
    'ridership_count': 'mean',
    'station_congestion_index': 'mean',
    'train_frequency': 'mean'
}).round(2)

print("\n🚇 Route-wise Analysis:")
print("="*80)
print(route_congestion.sort_values('station_congestion_index', ascending=False))

## 7️⃣ Export & Download

Download the generated dataset:

In [ ]:
# Download files (for Google Colab)
from google.colab import files

# Download CSV
files.download('output/delhi_metro_dataset.csv')

# Download plots
files.download('output/ridership_analysis.png')
files.download('output/ridership_heatmap.png')
files.download('output/weather_impact.png')

print("✅ Files ready for download!")

## 8️⃣ Generate MTA (New York) Dataset (Optional)

Uncomment to generate MTA data with real GTFS:

In [ ]:
# # Generate MTA dataset
# mta_df = generate_dataset(
#     config=MTA_CONFIG,
#     start_date=datetime(2024, 1, 1),
#     end_date=datetime(2024, 1, 31),
#     output_dir="output"
# )

# print("✅ MTA dataset generated!")
# mta_df.head()

## 9️⃣ Machine Learning Ready

The dataset is ready for ML models:

In [ ]:
# Example: Train a simple classifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Prepare features and target
feature_cols = [
    'hour_of_day', 'day_of_week', 'is_weekend', 'train_frequency',
    'temperature', 'rainfall', 'event_flag', 'rolling_demand_3h'
]

X = dmrc_df[feature_cols]
y = dmrc_df['crowd_level']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Evaluate
y_pred = rf_model.predict(X_test)
print("\n🤖 Model Performance:")
print("="*80)
print(classification_report(y_test, y_pred))

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n📊 Feature Importance:")
print(feature_importance)

## 🎉 Done!

You now have a production-quality metro transit dataset ready for:
- 🔮 Predictive modeling
- 📊 Time series analysis
- 🔍 Anomaly detection
- 📈 Business intelligence

### Next Steps:
1. ✅ Train advanced ML models (LSTM, Prophet, XGBoost)
2. ✅ Build real-time dashboards
3. ✅ Deploy predictions API
4. ✅ Integrate with mobile app

---

**Made with ❤️ by BheedMitra Team**